### check vocab match rate

In [1]:
#!/usr/bin/env python3
"""
Gene Vocabulary Alignment Script
Maps SEAAD Oli genes to scGPT vocabulary
"""

import json
import pandas as pd
import re
from collections import Counter

# =============================================================================
# PATHS - Update these if needed
# =============================================================================
SCGPT_VOCAB_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/scGPT/scgpt/tokenizer/default_gene_vocab.json"
SEAAD_GENES_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/seaad_oli_genes.csv"
OUTPUT_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_result.csv"

# =============================================================================
# Load Data
# =============================================================================
print("=" * 70)
print("STEP 1: Loading Data")
print("=" * 70)

# Load scGPT vocabulary
with open(SCGPT_VOCAB_PATH, 'r') as f:
    scgpt_vocab = json.load(f)

print(f"scGPT vocabulary size: {len(scgpt_vocab)}")

# Identify special tokens
special_tokens = [k for k in scgpt_vocab.keys() if k.startswith('<')]
gene_tokens = [k for k in scgpt_vocab.keys() if not k.startswith('<')]
print(f"  - Gene tokens: {len(gene_tokens)}")
print(f"  - Special tokens: {special_tokens}")

# Load SEAAD genes
seaad_df = pd.read_csv(SEAAD_GENES_PATH)
seaad_genes = seaad_df['gene_symbol'].tolist()
print(f"\nSEAAD Oli genes: {len(seaad_genes)}")
print(f"  - First 5: {seaad_genes[:5]}")
print(f"  - Last 5: {seaad_genes[-5:]}")

# =============================================================================
# Analysis: Gene Name Patterns
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2: Analyzing Gene Name Patterns")
print("=" * 70)

# Check for version suffixes in SEAAD
versioned_pattern = re.compile(r'\.\d+$')
seaad_versioned = [g for g in seaad_genes if versioned_pattern.search(g)]
print(f"\nSEAAD genes with version suffix (e.g., .1, .2): {len(seaad_versioned)}")
print(f"  - Examples: {seaad_versioned[:10]}")

# Check for version suffixes in scGPT
scgpt_versioned = [g for g in gene_tokens if versioned_pattern.search(g)]
print(f"\nscGPT genes with version suffix: {len(scgpt_versioned)}")
print(f"  - Examples: {scgpt_versioned[:10]}")

# =============================================================================
# Matching Strategy 1: Exact Match
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3: Exact Matching")
print("=" * 70)

scgpt_set = set(gene_tokens)
exact_matches = [g for g in seaad_genes if g in scgpt_set]
exact_unmatched = [g for g in seaad_genes if g not in scgpt_set]

print(f"Exact matches: {len(exact_matches)} / {len(seaad_genes)} ({100*len(exact_matches)/len(seaad_genes):.2f}%)")
print(f"Unmatched: {len(exact_unmatched)}")

# =============================================================================
# Matching Strategy 2: Strip Version Suffix
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Matching After Stripping Version Suffix")
print("=" * 70)

def strip_version(gene):
    """Remove version suffix like .1, .2 from gene names"""
    return versioned_pattern.sub('', gene)

# Create stripped version lookup for scGPT
scgpt_stripped = {strip_version(g): g for g in gene_tokens}
scgpt_stripped_set = set(scgpt_stripped.keys())

# Try matching unmatched genes after stripping
stripped_matches = {}
still_unmatched = []

for g in exact_unmatched:
    g_stripped = strip_version(g)
    if g_stripped in scgpt_stripped_set:
        stripped_matches[g] = scgpt_stripped[g_stripped]
    else:
        still_unmatched.append(g)

print(f"Additional matches after stripping version: {len(stripped_matches)}")
print(f"  - Examples: {list(stripped_matches.items())[:5]}")
print(f"Still unmatched: {len(still_unmatched)}")

# =============================================================================
# Matching Strategy 3: Case-Insensitive Match
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Case-Insensitive Matching (for remaining unmatched)")
print("=" * 70)

# Create lowercase lookup for scGPT
scgpt_lower = {g.lower(): g for g in gene_tokens}

case_matches = {}
final_unmatched = []

for g in still_unmatched:
    g_lower = g.lower()
    g_stripped_lower = strip_version(g).lower()
    
    if g_lower in scgpt_lower:
        case_matches[g] = scgpt_lower[g_lower]
    elif g_stripped_lower in scgpt_lower:
        case_matches[g] = scgpt_lower[g_stripped_lower]
    else:
        final_unmatched.append(g)

print(f"Additional matches (case-insensitive): {len(case_matches)}")
print(f"  - Examples: {list(case_matches.items())[:5]}")
print(f"Final unmatched: {len(final_unmatched)}")

# =============================================================================
# Summary Statistics
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: Summary Statistics")
print("=" * 70)

total_matched = len(exact_matches) + len(stripped_matches) + len(case_matches)
print(f"""
MATCHING SUMMARY:
─────────────────────────────────────────────────────────────────────
Total SEAAD genes:          {len(seaad_genes):,}
Total scGPT vocab genes:    {len(gene_tokens):,}

Match Results:
  ✓ Exact matches:          {len(exact_matches):,} ({100*len(exact_matches)/len(seaad_genes):.2f}%)
  ✓ Version-stripped:       {len(stripped_matches):,} ({100*len(stripped_matches)/len(seaad_genes):.2f}%)
  ✓ Case-insensitive:       {len(case_matches):,} ({100*len(case_matches)/len(seaad_genes):.2f}%)
  ─────────────────────────────────────────────────────────────────
  ✓ TOTAL MATCHED:          {total_matched:,} ({100*total_matched/len(seaad_genes):.2f}%)
  ✗ UNMATCHED (must drop):  {len(final_unmatched):,} ({100*len(final_unmatched)/len(seaad_genes):.2f}%)
─────────────────────────────────────────────────────────────────────
""")

# =============================================================================
# Analyze Unmatched Genes
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7: Analyzing Unmatched Genes")
print("=" * 70)

# Categorize unmatched genes by pattern
unmatched_categories = {
    'LINC (lncRNA)': [],
    'LOC (uncharacterized)': [],
    'AL/AC/AP (Ensembl novel)': [],
    'MIR (microRNA)': [],
    'SNORD/SNORA (snoRNA)': [],
    'Other': []
}

for g in final_unmatched:
    if g.startswith('LINC'):
        unmatched_categories['LINC (lncRNA)'].append(g)
    elif g.startswith('LOC'):
        unmatched_categories['LOC (uncharacterized)'].append(g)
    elif re.match(r'^(AL|AC|AP)\d+', g):
        unmatched_categories['AL/AC/AP (Ensembl novel)'].append(g)
    elif g.startswith('MIR'):
        unmatched_categories['MIR (microRNA)'].append(g)
    elif g.startswith('SNORD') or g.startswith('SNORA'):
        unmatched_categories['SNORD/SNORA (snoRNA)'].append(g)
    else:
        unmatched_categories['Other'].append(g)

print("Unmatched genes by category:")
for cat, genes in unmatched_categories.items():
    print(f"  {cat}: {len(genes)}")
    if len(genes) > 0 and len(genes) <= 10:
        print(f"    {genes}")
    elif len(genes) > 10:
        print(f"    Examples: {genes[:5]}")

# =============================================================================
# Create Mapping File
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8: Creating Mapping File")
print("=" * 70)

mapping_records = []

# Exact matches
for g in exact_matches:
    mapping_records.append({
        'seaad_symbol': g,
        'scgpt_symbol': g,
        'scgpt_token_id': scgpt_vocab[g],
        'match_type': 'exact',
        'status': 'matched'
    })

# Version-stripped matches
for seaad_g, scgpt_g in stripped_matches.items():
    mapping_records.append({
        'seaad_symbol': seaad_g,
        'scgpt_symbol': scgpt_g,
        'scgpt_token_id': scgpt_vocab[scgpt_g],
        'match_type': 'version_stripped',
        'status': 'matched'
    })

# Case-insensitive matches
for seaad_g, scgpt_g in case_matches.items():
    mapping_records.append({
        'seaad_symbol': seaad_g,
        'scgpt_symbol': scgpt_g,
        'scgpt_token_id': scgpt_vocab[scgpt_g],
        'match_type': 'case_insensitive',
        'status': 'matched'
    })

# Unmatched
for g in final_unmatched:
    mapping_records.append({
        'seaad_symbol': g,
        'scgpt_symbol': None,
        'scgpt_token_id': -1,
        'match_type': None,
        'status': 'unmatched'
    })

# Create DataFrame and save
mapping_df = pd.DataFrame(mapping_records)
mapping_df.to_csv(OUTPUT_MAPPING_PATH, index=False)
print(f"Saved mapping to: {OUTPUT_MAPPING_PATH}")

# =============================================================================
# Quick Validation
# =============================================================================
print("\n" + "=" * 70)
print("STEP 9: Validation")
print("=" * 70)

print(f"Mapping file shape: {mapping_df.shape}")
print(f"\nMatch type distribution:")
print(mapping_df['match_type'].value_counts(dropna=False))
print(f"\nStatus distribution:")
print(mapping_df['status'].value_counts())

# Sample of each type
print("\n--- Sample Exact Matches ---")
print(mapping_df[mapping_df['match_type'] == 'exact'].head(5).to_string(index=False))

print("\n--- Sample Version-Stripped Matches ---")
print(mapping_df[mapping_df['match_type'] == 'version_stripped'].head(5).to_string(index=False))

print("\n--- Sample Unmatched ---")
print(mapping_df[mapping_df['status'] == 'unmatched'].head(10).to_string(index=False))

print("\n" + "=" * 70)
print("DONE! Gene vocabulary alignment complete.")
print("=" * 70)

STEP 1: Loading Data
scGPT vocabulary size: 48292
  - Gene tokens: 48292
  - Special tokens: []

SEAAD Oli genes: 36601
  - First 5: ['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3']
  - Last 5: ['AC141272.1', 'AC023491.2', 'AC007325.1', 'AC007325.4', 'AC007325.2']

STEP 2: Analyzing Gene Name Patterns

SEAAD genes with version suffix (e.g., .1, .2): 12554
  - Examples: ['AL627309.1', 'AL627309.3', 'AL627309.2', 'AL627309.5', 'AL627309.4', 'AP006222.2', 'AL732372.1', 'AC114498.1', 'AL669831.2', 'AL645608.6']

scGPT genes with version suffix: 0
  - Examples: []

STEP 3: Exact Matching
Exact matches: 23536 / 36601 (64.30%)
Unmatched: 13065

STEP 4: Matching After Stripping Version Suffix
Additional matches after stripping version: 0
  - Examples: []
Still unmatched: 13065

STEP 5: Case-Insensitive Matching (for remaining unmatched)
Additional matches (case-insensitive): 0
  - Examples: []
Final unmatched: 13065

STEP 6: Summary Statistics

MATCHING SUMMARY:
────────────────

In [2]:
#!/usr/bin/env python3
"""
Compare gene vocabulary alignment with default_census_vocab.json
"""

import json
import pandas as pd
import numpy as np
import re

# =============================================================================
# PATHS
# =============================================================================
CENSUS_VOCAB_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/scGPT/scgpt/tokenizer/default_census_vocab.json"
ORIGINAL_VOCAB_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/scGPT/scgpt/tokenizer/default_gene_vocab.json"
SEAAD_GENES_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/seaad_oli_genes.csv"
OUTPUT_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_census_vocab.csv"

# =============================================================================
# Load vocabularies
# =============================================================================
print("=" * 70)
print("Step 1: Loading vocabularies")
print("=" * 70)

# Census vocab
with open(CENSUS_VOCAB_PATH, 'r') as f:
    census_vocab = json.load(f)
print(f"Census vocab size: {len(census_vocab)}")

# Original vocab (for comparison)
with open(ORIGINAL_VOCAB_PATH, 'r') as f:
    original_vocab = json.load(f)
print(f"Original vocab size: {len(original_vocab)}")

# SEAAD genes
seaad_df = pd.read_csv(SEAAD_GENES_PATH)
seaad_genes = seaad_df['gene_symbol'].tolist()
print(f"SEAAD genes: {len(seaad_genes)}")

# =============================================================================
# Compare vocab structures
# =============================================================================
print("\n" + "=" * 70)
print("Step 2: Comparing vocabularies")
print("=" * 70)

# Special tokens
census_special = [k for k in census_vocab.keys() if k.startswith('<')]
original_special = [k for k in original_vocab.keys() if k.startswith('<')]

print(f"\nCensus special tokens: {census_special}")
print(f"Original special tokens: {original_special}")

# Gene tokens
census_genes = set(k for k in census_vocab.keys() if not k.startswith('<'))
original_genes = set(k for k in original_vocab.keys() if not k.startswith('<'))

print(f"\nCensus gene tokens: {len(census_genes)}")
print(f"Original gene tokens: {len(original_genes)}")

# Overlap between vocabs
vocab_overlap = census_genes & original_genes
census_only = census_genes - original_genes
original_only = original_genes - census_genes

print(f"\nOverlap: {len(vocab_overlap)}")
print(f"Census only: {len(census_only)}")
print(f"Original only: {len(original_only)}")

# Sample census-only genes
print(f"\nSample census-only genes: {list(census_only)[:20]}")

# Check for versioned genes in census vocab
versioned_pattern = re.compile(r'\.\d+$')
census_versioned = [g for g in census_genes if versioned_pattern.search(g)]
print(f"\nCensus versioned genes (e.g., .1, .2): {len(census_versioned)}")
print(f"  Examples: {census_versioned[:10]}")

# =============================================================================
# Match SEAAD genes against Census vocab
# =============================================================================
print("\n" + "=" * 70)
print("Step 3: Matching SEAAD genes to Census vocab")
print("=" * 70)

# Exact matching
exact_matches = [g for g in seaad_genes if g in census_genes]
exact_unmatched = [g for g in seaad_genes if g not in census_genes]

print(f"Exact matches: {len(exact_matches)} / {len(seaad_genes)} ({100*len(exact_matches)/len(seaad_genes):.2f}%)")
print(f"Unmatched: {len(exact_unmatched)}")

# =============================================================================
# Comparison: Census vs Original
# =============================================================================
print("\n" + "=" * 70)
print("Step 4: Comparison - Census vs Original vocab")
print("=" * 70)

# Original matching (from before)
original_matches = [g for g in seaad_genes if g in original_genes]

print(f"""
MATCHING COMPARISON:
─────────────────────────────────────────────────────────────────────
                        Original Vocab      Census Vocab
─────────────────────────────────────────────────────────────────────
Vocab size:             {len(original_genes):,}              {len(census_genes):,}
SEAAD matches:          {len(original_matches):,} ({100*len(original_matches)/len(seaad_genes):.1f}%)        {len(exact_matches):,} ({100*len(exact_matches)/len(seaad_genes):.1f}%)
Improvement:            -                   {len(exact_matches) - len(original_matches):+,} genes
─────────────────────────────────────────────────────────────────────
""")

# =============================================================================
# Analyze newly matched genes
# =============================================================================
print("\n" + "=" * 70)
print("Step 5: Analyzing newly matched genes (Census but not Original)")
print("=" * 70)

original_matched_set = set(original_matches)
newly_matched = [g for g in exact_matches if g not in original_matched_set]

print(f"Newly matched genes: {len(newly_matched)}")
print(f"Examples: {newly_matched[:30]}")

# Categorize newly matched
categories = {
    'AL/AC/AP (novel)': lambda g: bool(re.match(r'^(AL|AC|AP)\d+', g)),
    'Versioned (.N)': lambda g: bool(versioned_pattern.search(g)),
    'LINC (lncRNA)': lambda g: g.startswith('LINC'),
    'Other': lambda g: True
}

print("\nNewly matched genes by category:")
for cat_name, cat_func in categories.items():
    if cat_name == 'Other':
        other_count = len([g for g in newly_matched if not any(
            categories[c](g) for c in categories if c != 'Other'
        )])
        print(f"  {cat_name}: {other_count}")
    else:
        count = sum(1 for g in newly_matched if cat_func(g))
        print(f"  {cat_name}: {count}")

# =============================================================================
# Analyze still-unmatched genes
# =============================================================================
print("\n" + "=" * 70)
print("Step 6: Still unmatched genes with Census vocab")
print("=" * 70)

still_unmatched = exact_unmatched
print(f"Still unmatched: {len(still_unmatched)}")

# Categorize
print("\nStill unmatched by category:")
for cat_name, cat_func in list(categories.items())[:-1]:  # Skip 'Other'
    count = sum(1 for g in still_unmatched if cat_func(g))
    print(f"  {cat_name}: {count}")

# Well-annotated unmatched
well_annotated = [g for g in still_unmatched if not (
    re.match(r'^(AL|AC|AP)\d+', g) or 
    versioned_pattern.search(g) or
    g.startswith('LINC') or
    g.startswith('LOC') or
    g.startswith('MIR')
)]
print(f"  Well-annotated: {len(well_annotated)}")
print(f"    Examples: {well_annotated[:20]}")

# =============================================================================
# Create mapping file
# =============================================================================
print("\n" + "=" * 70)
print("Step 7: Creating mapping file")
print("=" * 70)

mapping_records = []
for g in seaad_genes:
    if g in census_genes:
        mapping_records.append({
            'seaad_symbol': g,
            'scgpt_symbol': g,
            'scgpt_token_id': census_vocab[g],
            'match_type': 'exact',
            'status': 'matched'
        })
    else:
        mapping_records.append({
            'seaad_symbol': g,
            'scgpt_symbol': None,
            'scgpt_token_id': -1,
            'match_type': None,
            'status': 'unmatched'
        })

mapping_df = pd.DataFrame(mapping_records)
mapping_df.to_csv(OUTPUT_MAPPING_PATH, index=False)
print(f"Saved to: {OUTPUT_MAPPING_PATH}")

# =============================================================================
# Final Summary
# =============================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"""
CENSUS VOCAB RESULTS:
─────────────────────────────────────────────────────────────────────
Total SEAAD genes:        {len(seaad_genes):,}
Matched with Census:      {len(exact_matches):,} ({100*len(exact_matches)/len(seaad_genes):.1f}%)
Unmatched:                {len(exact_unmatched):,} ({100*len(exact_unmatched)/len(seaad_genes):.1f}%)

IMPROVEMENT OVER ORIGINAL:
  Original match rate:    {100*len(original_matches)/len(seaad_genes):.1f}%
  Census match rate:      {100*len(exact_matches)/len(seaad_genes):.1f}%
  Improvement:            {100*(len(exact_matches)-len(original_matches))/len(seaad_genes):.1f}% more genes matched
─────────────────────────────────────────────────────────────────────
""")

if len(exact_matches) > len(original_matches):
    print("✓ RECOMMENDATION: Use Census vocab for better coverage!")
else:
    print("→ Census vocab doesn't improve coverage significantly")

Step 1: Loading vocabularies
Census vocab size: 60694
Original vocab size: 48292
SEAAD genes: 36601

Step 2: Comparing vocabularies

Census special tokens: []
Original special tokens: []

Census gene tokens: 60694
Original gene tokens: 48292

Overlap: 38972
Census only: 21722
Original only: 9320

Sample census-only genes: ['RP11-332L11.1', 'RP11-151H2.3', 'RP11-359P18.2', 'CTD-2308B18.4', 'RP1-317E23.3', 'RP11-574M7.1', 'AC079790.2', 'RP11-302L19.5', 'RP11-468H14.2', 'RP11-94H18.1', 'RP11-382A20.7', 'Y_RNA_ENSG00000202368', 'RP11-807H22.6', 'CTD-2007F2.1', 'RP11-735A19.3', 'RP11-126O22.8', 'RP11-779O18.2', 'RP1-134E15.3', 'RP3-468B3.6', 'RP11-359K18.3']

Census versioned genes (e.g., .1, .2): 19964
  Examples: ['RP11-332L11.1', 'RP11-151H2.3', 'RP11-359P18.2', 'RP1-317E23.3', 'CTD-2308B18.4', 'AC079790.2', 'RP11-574M7.1', 'RP11-302L19.5', 'RP11-468H14.2', 'RP11-94H18.1']

Step 3: Matching SEAAD genes to Census vocab
Exact matches: 24285 / 36601 (66.35%)
Unmatched: 12316

Step 4: Compar

### check importance of unmatched genes

In [1]:
#!/usr/bin/env python3
"""
ULTRA MEMORY-EFFICIENT: Read cell data directly from HDF5 file
Never loads full indices/data arrays
"""

import h5py
import numpy as np
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================
DATA_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/SEAAD_A9_RNAseq_DREAM.2025-07-15.h5ad"
MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_result.csv"

SAMPLE_SIZE = 5000  # Sample this many Oli cells

# =============================================================================
# Step 1: Get matched gene indices
# =============================================================================
print("=" * 70)
print("Step 1: Loading gene mapping")
print("=" * 70)

mapping_df = pd.read_csv(MAPPING_PATH)

with h5py.File(DATA_PATH, 'r') as f:
    gene_names = f['var']['_index'][:].astype(str)

matched_set = set(mapping_df[mapping_df['status'] == 'matched']['seaad_symbol'])
matched_gene_indices = set(i for i, g in enumerate(gene_names) if g in matched_set)

print(f"Total genes: {len(gene_names)}")
print(f"Matched gene indices: {len(matched_gene_indices)}")

# =============================================================================
# Step 2: Find Oli cell indices
# =============================================================================
print("\n" + "=" * 70)
print("Step 2: Finding Oli cells")
print("=" * 70)

with h5py.File(DATA_PATH, 'r') as f:
    # Load Subclass categorical
    categories = f['obs']['Subclass']['categories'][:].astype(str)
    codes = f['obs']['Subclass']['codes'][:]
    
    # Find Oligodendrocyte category index
    oli_cat_idx = None
    for i, cat in enumerate(categories):
        if 'oligodendrocyte' in cat.lower():
            oli_cat_idx = i
            print(f"Found: '{cat}' at index {i}")
            break
    
    # Find cells with this category
    oli_cell_indices = np.where(codes == oli_cat_idx)[0]
    print(f"Total Oli cells: {len(oli_cell_indices):,}")

# =============================================================================
# Step 3: Sample and analyze Oli cells
# =============================================================================
print("\n" + "=" * 70)
print(f"Step 3: Analyzing {min(SAMPLE_SIZE, len(oli_cell_indices))} sampled Oli cells")
print("=" * 70)

# Sample
if len(oli_cell_indices) > SAMPLE_SIZE:
    np.random.seed(42)
    sampled_oli = np.random.choice(oli_cell_indices, SAMPLE_SIZE, replace=False)
else:
    sampled_oli = oli_cell_indices

sampled_oli = np.sort(sampled_oli)
print(f"Sampled {len(sampled_oli)} Oli cells")

# Process cells one by one, reading directly from HDF5
matched_expr_list = []
unmatched_expr_list = []

with h5py.File(DATA_PATH, 'r') as f:
    indptr = f['X']['indptr'][:]  # This is small - just n_cells+1 integers
    indices_dset = f['X']['indices']  # Don't load, just reference
    data_dset = f['X']['data']
    
    for i, cell_idx in enumerate(sampled_oli):
        if i % 1000 == 0:
            print(f"  Processing cell {i}/{len(sampled_oli)}...")
        
        # Get slice for this cell
        start = indptr[cell_idx]
        end = indptr[cell_idx + 1]
        
        # Read ONLY this cell's data from HDF5
        cell_gene_idx = indices_dset[start:end]
        cell_expr = data_dset[start:end]
        
        # Calculate matched vs unmatched expression
        matched_expr = 0.0
        unmatched_expr = 0.0
        
        for g_idx, expr in zip(cell_gene_idx, cell_expr):
            if g_idx in matched_gene_indices:
                matched_expr += expr
            else:
                unmatched_expr += expr
        
        matched_expr_list.append(matched_expr)
        unmatched_expr_list.append(unmatched_expr)

print("Done processing!")

# =============================================================================
# Step 4: Results
# =============================================================================
print("\n" + "=" * 70)
print("RESULTS: Unmatched Gene Activity in Oli Cells")
print("=" * 70)

matched_expr = np.array(matched_expr_list)
unmatched_expr = np.array(unmatched_expr_list)
total_expr = matched_expr + unmatched_expr

pct_matched = 100 * matched_expr / (total_expr + 1e-10)
pct_unmatched = 100 * unmatched_expr / (total_expr + 1e-10)

print(f"""
EXPRESSION CONTRIBUTION IN OLI CELLS ({len(sampled_oli):,} cells):
─────────────────────────────────────────────────────────────────────
                              Mean      Median    Min       Max
─────────────────────────────────────────────────────────────────────
% from MATCHED genes:         {pct_matched.mean():.2f}%    {np.median(pct_matched):.2f}%    {pct_matched.min():.2f}%    {pct_matched.max():.2f}%
% from UNMATCHED genes:       {pct_unmatched.mean():.2f}%     {np.median(pct_unmatched):.2f}%     {pct_unmatched.min():.2f}%     {pct_unmatched.max():.2f}%
─────────────────────────────────────────────────────────────────────
""")

# Percentiles
print("Percentile distribution of UNMATCHED expression %:")
for p in [5, 25, 50, 75, 95]:
    print(f"  {p}th: {np.percentile(pct_unmatched, p):.2f}%")

# Verdict
print("\n" + "=" * 70)
avg_unmatched = pct_unmatched.mean()
if avg_unmatched < 5:
    print(f"✓ SAFE: Unmatched genes = {avg_unmatched:.2f}% of expression")
    print("  scGPT will capture >95% of biological signal")
elif avg_unmatched < 15:
    print(f"⚠ CAUTION: Unmatched genes = {avg_unmatched:.2f}% of expression")
else:
    print(f"✗ SIGNIFICANT LOSS: Unmatched genes = {avg_unmatched:.2f}%")
print("=" * 70)

Step 1: Loading gene mapping
Total genes: 36601
Matched gene indices: 23536

Step 2: Finding Oli cells
Found: 'Oligodendrocyte' at index 20
Total Oli cells: 142,064

Step 3: Analyzing 5000 sampled Oli cells
Sampled 5000 Oli cells
  Processing cell 0/5000...
  Processing cell 1000/5000...
  Processing cell 2000/5000...
  Processing cell 3000/5000...
  Processing cell 4000/5000...
Done processing!

RESULTS: Unmatched Gene Activity in Oli Cells

EXPRESSION CONTRIBUTION IN OLI CELLS (5,000 cells):
─────────────────────────────────────────────────────────────────────
                              Mean      Median    Min       Max
─────────────────────────────────────────────────────────────────────
% from MATCHED genes:         93.63%    93.77%    89.61%    96.79%
% from UNMATCHED genes:       6.37%     6.23%     3.21%     10.39%
─────────────────────────────────────────────────────────────────────

Percentile distribution of UNMATCHED expression %:
  5th: 4.81%
  25th: 5.56%
  50th: 6.23%


In [3]:
#!/usr/bin/env python3
"""
Check unmatched gene importance using Census vocab mapping
"""

import h5py
import numpy as np
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================
DATA_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/SEAAD_A9_RNAseq_DREAM.2025-07-15.h5ad"
CENSUS_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_census_vocab.csv"

SAMPLE_SIZE = 5000  # Sample this many Oli cells

# =============================================================================
# Step 1: Get matched gene indices (Census vocab)
# =============================================================================
print("=" * 70)
print("Step 1: Loading Census vocab gene mapping")
print("=" * 70)

mapping_df = pd.read_csv(CENSUS_MAPPING_PATH)

with h5py.File(DATA_PATH, 'r') as f:
    gene_names = f['var']['_index'][:].astype(str)

matched_set = set(mapping_df[mapping_df['status'] == 'matched']['seaad_symbol'])
matched_gene_indices = set(i for i, g in enumerate(gene_names) if g in matched_set)

print(f"Total genes: {len(gene_names)}")
print(f"Matched genes (Census): {len(matched_gene_indices)}")
print(f"Unmatched genes: {len(gene_names) - len(matched_gene_indices)}")

# =============================================================================
# Step 2: Find Oli cell indices
# =============================================================================
print("\n" + "=" * 70)
print("Step 2: Finding Oli cells")
print("=" * 70)

with h5py.File(DATA_PATH, 'r') as f:
    categories = f['obs']['Subclass']['categories'][:].astype(str)
    codes = f['obs']['Subclass']['codes'][:]
    
    oli_cat_idx = None
    for i, cat in enumerate(categories):
        if 'oligodendrocyte' in cat.lower():
            oli_cat_idx = i
            break
    
    oli_cell_indices = np.where(codes == oli_cat_idx)[0]
    print(f"Total Oli cells: {len(oli_cell_indices):,}")

# =============================================================================
# Step 3: Sample and analyze Oli cells
# =============================================================================
print("\n" + "=" * 70)
print(f"Step 3: Analyzing {min(SAMPLE_SIZE, len(oli_cell_indices))} Oli cells")
print("=" * 70)

if len(oli_cell_indices) > SAMPLE_SIZE:
    np.random.seed(42)
    sampled_oli = np.random.choice(oli_cell_indices, SAMPLE_SIZE, replace=False)
else:
    sampled_oli = oli_cell_indices

sampled_oli = np.sort(sampled_oli)

matched_expr_list = []
unmatched_expr_list = []

with h5py.File(DATA_PATH, 'r') as f:
    indptr = f['X']['indptr'][:]
    indices_dset = f['X']['indices']
    data_dset = f['X']['data']
    
    for i, cell_idx in enumerate(sampled_oli):
        if i % 1000 == 0:
            print(f"  Processing cell {i}/{len(sampled_oli)}...")
        
        start = indptr[cell_idx]
        end = indptr[cell_idx + 1]
        
        cell_gene_idx = indices_dset[start:end]
        cell_expr = data_dset[start:end]
        
        matched_expr = 0.0
        unmatched_expr = 0.0
        
        for g_idx, expr in zip(cell_gene_idx, cell_expr):
            if g_idx in matched_gene_indices:
                matched_expr += expr
            else:
                unmatched_expr += expr
        
        matched_expr_list.append(matched_expr)
        unmatched_expr_list.append(unmatched_expr)

print("Done!")

# =============================================================================
# Step 4: Results
# =============================================================================
print("\n" + "=" * 70)
print("RESULTS: Census Vocab - Unmatched Gene Activity in Oli Cells")
print("=" * 70)

matched_expr = np.array(matched_expr_list)
unmatched_expr = np.array(unmatched_expr_list)
total_expr = matched_expr + unmatched_expr

pct_matched = 100 * matched_expr / (total_expr + 1e-10)
pct_unmatched = 100 * unmatched_expr / (total_expr + 1e-10)

print(f"""
EXPRESSION CONTRIBUTION IN OLI CELLS ({len(sampled_oli):,} cells):
─────────────────────────────────────────────────────────────────────
                              Mean      Median    Min       Max
─────────────────────────────────────────────────────────────────────
% from MATCHED genes:         {pct_matched.mean():.2f}%    {np.median(pct_matched):.2f}%    {pct_matched.min():.2f}%    {pct_matched.max():.2f}%
% from UNMATCHED genes:       {pct_unmatched.mean():.2f}%     {np.median(pct_unmatched):.2f}%     {pct_unmatched.min():.2f}%     {pct_unmatched.max():.2f}%
─────────────────────────────────────────────────────────────────────
""")

# Percentiles
print("Percentile distribution of UNMATCHED expression %:")
for p in [5, 25, 50, 75, 95]:
    print(f"  {p}th: {np.percentile(pct_unmatched, p):.2f}%")

# =============================================================================
# Comparison with Original vocab
# =============================================================================
print("\n" + "=" * 70)
print("COMPARISON: Census vs Original Vocab")
print("=" * 70)

# Original results (from previous run)
orig_matched_pct = 93.63
orig_unmatched_pct = 6.37

print(f"""
─────────────────────────────────────────────────────────────────────
                        Original Vocab      Census Vocab
─────────────────────────────────────────────────────────────────────
Gene match rate:        64.3%               66.4%
Expression matched:     {orig_matched_pct:.2f}%             {pct_matched.mean():.2f}%
Expression unmatched:   {orig_unmatched_pct:.2f}%              {pct_unmatched.mean():.2f}%
─────────────────────────────────────────────────────────────────────
""")

# Verdict
print("\n" + "=" * 70)
avg_unmatched = pct_unmatched.mean()
if avg_unmatched < 5:
    print(f"✓ SAFE: Unmatched genes = {avg_unmatched:.2f}% of expression")
    print("  scGPT will capture >95% of biological signal")
elif avg_unmatched < 10:
    print(f"✓ GOOD: Unmatched genes = {avg_unmatched:.2f}% of expression")
    print("  scGPT will capture >90% of biological signal")
elif avg_unmatched < 15:
    print(f"⚠ CAUTION: Unmatched genes = {avg_unmatched:.2f}% of expression")
else:
    print(f"✗ SIGNIFICANT LOSS: Unmatched genes = {avg_unmatched:.2f}%")
print("=" * 70)

Step 1: Loading Census vocab gene mapping
Total genes: 36601
Matched genes (Census): 24285
Unmatched genes: 12316

Step 2: Finding Oli cells
Total Oli cells: 142,064

Step 3: Analyzing 5000 Oli cells
  Processing cell 0/5000...
  Processing cell 1000/5000...
  Processing cell 2000/5000...
  Processing cell 3000/5000...
  Processing cell 4000/5000...
Done!

RESULTS: Census Vocab - Unmatched Gene Activity in Oli Cells

EXPRESSION CONTRIBUTION IN OLI CELLS (5,000 cells):
─────────────────────────────────────────────────────────────────────
                              Mean      Median    Min       Max
─────────────────────────────────────────────────────────────────────
% from MATCHED genes:         94.21%    94.35%    90.21%    97.02%
% from UNMATCHED genes:       5.79%     5.65%     2.98%     9.79%
─────────────────────────────────────────────────────────────────────

Percentile distribution of UNMATCHED expression %:
  5th: 4.32%
  25th: 5.02%
  50th: 5.65%
  75th: 6.48%
  95th: 7.65%

In [4]:
#!/usr/bin/env python3
"""
Analyze ALL Oli cells (142,064) with Census vocab mapping
"""

import h5py
import numpy as np
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================
DATA_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/SEAAD_A9_RNAseq_DREAM.2025-07-15.h5ad"
CENSUS_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_census_vocab.csv"

# =============================================================================
# Step 1: Get matched gene indices (Census vocab)
# =============================================================================
print("=" * 70)
print("Step 1: Loading Census vocab gene mapping")
print("=" * 70)

mapping_df = pd.read_csv(CENSUS_MAPPING_PATH)

with h5py.File(DATA_PATH, 'r') as f:
    gene_names = f['var']['_index'][:].astype(str)

matched_set = set(mapping_df[mapping_df['status'] == 'matched']['seaad_symbol'])
matched_gene_indices = set(i for i, g in enumerate(gene_names) if g in matched_set)

print(f"Total genes: {len(gene_names)}")
print(f"Matched genes (Census): {len(matched_gene_indices)}")

# =============================================================================
# Step 2: Find ALL Oli cell indices
# =============================================================================
print("\n" + "=" * 70)
print("Step 2: Finding ALL Oli cells")
print("=" * 70)

with h5py.File(DATA_PATH, 'r') as f:
    categories = f['obs']['Subclass']['categories'][:].astype(str)
    codes = f['obs']['Subclass']['codes'][:]
    
    oli_cat_idx = None
    for i, cat in enumerate(categories):
        if 'oligodendrocyte' in cat.lower():
            oli_cat_idx = i
            break
    
    oli_cell_indices = np.where(codes == oli_cat_idx)[0]
    print(f"Total Oli cells to analyze: {len(oli_cell_indices):,}")

# =============================================================================
# Step 3: Analyze ALL Oli cells
# =============================================================================
print("\n" + "=" * 70)
print(f"Step 3: Analyzing ALL {len(oli_cell_indices):,} Oli cells")
print("=" * 70)

matched_expr_list = []
unmatched_expr_list = []

with h5py.File(DATA_PATH, 'r') as f:
    indptr = f['X']['indptr'][:]
    indices_dset = f['X']['indices']
    data_dset = f['X']['data']
    
    total_cells = len(oli_cell_indices)
    
    for i, cell_idx in enumerate(oli_cell_indices):
        if i % 10000 == 0:
            print(f"  Processing cell {i:,}/{total_cells:,} ({100*i/total_cells:.1f}%)...")
        
        start = indptr[cell_idx]
        end = indptr[cell_idx + 1]
        
        cell_gene_idx = indices_dset[start:end]
        cell_expr = data_dset[start:end]
        
        matched_expr = 0.0
        unmatched_expr = 0.0
        
        for g_idx, expr in zip(cell_gene_idx, cell_expr):
            if g_idx in matched_gene_indices:
                matched_expr += expr
            else:
                unmatched_expr += expr
        
        matched_expr_list.append(matched_expr)
        unmatched_expr_list.append(unmatched_expr)

print(f"  Processing cell {total_cells:,}/{total_cells:,} (100%)... Done!")

# =============================================================================
# Step 4: Results
# =============================================================================
print("\n" + "=" * 70)
print("RESULTS: Census Vocab - ALL Oli Cells")
print("=" * 70)

matched_expr = np.array(matched_expr_list)
unmatched_expr = np.array(unmatched_expr_list)
total_expr = matched_expr + unmatched_expr

pct_matched = 100 * matched_expr / (total_expr + 1e-10)
pct_unmatched = 100 * unmatched_expr / (total_expr + 1e-10)

print(f"""
EXPRESSION CONTRIBUTION IN ALL OLI CELLS ({len(oli_cell_indices):,} cells):
─────────────────────────────────────────────────────────────────────
                              Mean      Median    Min       Max       Std
─────────────────────────────────────────────────────────────────────
% from MATCHED genes:         {pct_matched.mean():.2f}%    {np.median(pct_matched):.2f}%    {pct_matched.min():.2f}%    {pct_matched.max():.2f}%    {pct_matched.std():.2f}%
% from UNMATCHED genes:       {pct_unmatched.mean():.2f}%     {np.median(pct_unmatched):.2f}%     {pct_unmatched.min():.2f}%     {pct_unmatched.max():.2f}%     {pct_unmatched.std():.2f}%
─────────────────────────────────────────────────────────────────────
""")

# Percentiles
print("Percentile distribution of UNMATCHED expression %:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p}th: {np.percentile(pct_unmatched, p):.2f}%")

# How many cells have high unmatched %?
print("\nCells by unmatched expression level:")
thresholds = [5, 7.5, 10, 15, 20]
for thresh in thresholds:
    count = (pct_unmatched > thresh).sum()
    pct = 100 * count / len(pct_unmatched)
    print(f"  >{thresh}%: {count:,} cells ({pct:.1f}%)")

# =============================================================================
# Summary
# =============================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

avg_unmatched = pct_unmatched.mean()
print(f"""
Census Vocab Gene Mapping Results (ALL {len(oli_cell_indices):,} Oli cells):
─────────────────────────────────────────────────────────────────────
Gene coverage:            24,285 / 36,601 = 66.4%
Expression captured:      {pct_matched.mean():.2f}%
Expression lost:          {pct_unmatched.mean():.2f}%
─────────────────────────────────────────────────────────────────────
""")

if avg_unmatched < 5:
    print(f"✓ EXCELLENT: Only {avg_unmatched:.2f}% expression from unmatched genes")
elif avg_unmatched < 7:
    print(f"✓ GOOD: {avg_unmatched:.2f}% expression from unmatched genes")
elif avg_unmatched < 10:
    print(f"✓ ACCEPTABLE: {avg_unmatched:.2f}% expression from unmatched genes")
else:
    print(f"⚠ CAUTION: {avg_unmatched:.2f}% expression from unmatched genes")

print("\n→ Proceed with scGPT finetuning using Census vocab")
print("=" * 70)

Step 1: Loading Census vocab gene mapping
Total genes: 36601
Matched genes (Census): 24285

Step 2: Finding ALL Oli cells
Total Oli cells to analyze: 142,064

Step 3: Analyzing ALL 142,064 Oli cells
  Processing cell 0/142,064 (0.0%)...
  Processing cell 10,000/142,064 (7.0%)...
  Processing cell 20,000/142,064 (14.1%)...
  Processing cell 30,000/142,064 (21.1%)...
  Processing cell 40,000/142,064 (28.2%)...
  Processing cell 50,000/142,064 (35.2%)...
  Processing cell 60,000/142,064 (42.2%)...
  Processing cell 70,000/142,064 (49.3%)...
  Processing cell 80,000/142,064 (56.3%)...
  Processing cell 90,000/142,064 (63.4%)...
  Processing cell 100,000/142,064 (70.4%)...
  Processing cell 110,000/142,064 (77.4%)...
  Processing cell 120,000/142,064 (84.5%)...
  Processing cell 130,000/142,064 (91.5%)...
  Processing cell 140,000/142,064 (98.5%)...
  Processing cell 142,064/142,064 (100%)... Done!

RESULTS: Census Vocab - ALL Oli Cells

EXPRESSION CONTRIBUTION IN ALL OLI CELLS (142,064 cel

In [5]:
#!/usr/bin/env python3
"""
Analyze ALL Oli cells (142,064) with Original default_gene_vocab mapping
"""

import h5py
import numpy as np
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================
DATA_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/SEAAD_A9_RNAseq_DREAM.2025-07-15.h5ad"
ORIGINAL_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_result.csv"

# =============================================================================
# Step 1: Get matched gene indices (Original vocab)
# =============================================================================
print("=" * 70)
print("Step 1: Loading Original (default_gene_vocab) mapping")
print("=" * 70)

mapping_df = pd.read_csv(ORIGINAL_MAPPING_PATH)

with h5py.File(DATA_PATH, 'r') as f:
    gene_names = f['var']['_index'][:].astype(str)

matched_set = set(mapping_df[mapping_df['status'] == 'matched']['seaad_symbol'])
matched_gene_indices = set(i for i, g in enumerate(gene_names) if g in matched_set)

print(f"Total genes: {len(gene_names)}")
print(f"Matched genes (Original): {len(matched_gene_indices)}")

# =============================================================================
# Step 2: Find ALL Oli cell indices
# =============================================================================
print("\n" + "=" * 70)
print("Step 2: Finding ALL Oli cells")
print("=" * 70)

with h5py.File(DATA_PATH, 'r') as f:
    categories = f['obs']['Subclass']['categories'][:].astype(str)
    codes = f['obs']['Subclass']['codes'][:]
    
    oli_cat_idx = None
    for i, cat in enumerate(categories):
        if 'oligodendrocyte' in cat.lower():
            oli_cat_idx = i
            break
    
    oli_cell_indices = np.where(codes == oli_cat_idx)[0]
    print(f"Total Oli cells to analyze: {len(oli_cell_indices):,}")

# =============================================================================
# Step 3: Analyze ALL Oli cells
# =============================================================================
print("\n" + "=" * 70)
print(f"Step 3: Analyzing ALL {len(oli_cell_indices):,} Oli cells")
print("=" * 70)

matched_expr_list = []
unmatched_expr_list = []

with h5py.File(DATA_PATH, 'r') as f:
    indptr = f['X']['indptr'][:]
    indices_dset = f['X']['indices']
    data_dset = f['X']['data']
    
    total_cells = len(oli_cell_indices)
    
    for i, cell_idx in enumerate(oli_cell_indices):
        if i % 10000 == 0:
            print(f"  Processing cell {i:,}/{total_cells:,} ({100*i/total_cells:.1f}%)...")
        
        start = indptr[cell_idx]
        end = indptr[cell_idx + 1]
        
        cell_gene_idx = indices_dset[start:end]
        cell_expr = data_dset[start:end]
        
        matched_expr = 0.0
        unmatched_expr = 0.0
        
        for g_idx, expr in zip(cell_gene_idx, cell_expr):
            if g_idx in matched_gene_indices:
                matched_expr += expr
            else:
                unmatched_expr += expr
        
        matched_expr_list.append(matched_expr)
        unmatched_expr_list.append(unmatched_expr)

print(f"  Processing cell {total_cells:,}/{total_cells:,} (100%)... Done!")

# =============================================================================
# Step 4: Results
# =============================================================================
print("\n" + "=" * 70)
print("RESULTS: Original Vocab (default_gene_vocab) - ALL Oli Cells")
print("=" * 70)

matched_expr = np.array(matched_expr_list)
unmatched_expr = np.array(unmatched_expr_list)
total_expr = matched_expr + unmatched_expr

pct_matched = 100 * matched_expr / (total_expr + 1e-10)
pct_unmatched = 100 * unmatched_expr / (total_expr + 1e-10)

print(f"""
EXPRESSION CONTRIBUTION IN ALL OLI CELLS ({len(oli_cell_indices):,} cells):
─────────────────────────────────────────────────────────────────────
                              Mean      Median    Min       Max       Std
─────────────────────────────────────────────────────────────────────
% from MATCHED genes:         {pct_matched.mean():.2f}%    {np.median(pct_matched):.2f}%    {pct_matched.min():.2f}%    {pct_matched.max():.2f}%    {pct_matched.std():.2f}%
% from UNMATCHED genes:       {pct_unmatched.mean():.2f}%     {np.median(pct_unmatched):.2f}%     {pct_unmatched.min():.2f}%     {pct_unmatched.max():.2f}%     {pct_unmatched.std():.2f}%
─────────────────────────────────────────────────────────────────────
""")

# Percentiles
print("Percentile distribution of UNMATCHED expression %:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p}th: {np.percentile(pct_unmatched, p):.2f}%")

# How many cells have high unmatched %?
print("\nCells by unmatched expression level:")
thresholds = [5, 7.5, 10, 15, 20]
for thresh in thresholds:
    count = (pct_unmatched > thresh).sum()
    pct = 100 * count / len(pct_unmatched)
    print(f"  >{thresh}%: {count:,} cells ({pct:.1f}%)")

# =============================================================================
# Comparison with Census vocab
# =============================================================================
print("\n" + "=" * 70)
print("COMPARISON: Original vs Census Vocab (ALL 142,064 Oli Cells)")
print("=" * 70)

# Census results (from previous run)
census_matched_genes = 24285
census_expr_matched = 94.22
census_expr_unmatched = 5.78

print(f"""
─────────────────────────────────────────────────────────────────────
                        Original Vocab      Census Vocab
─────────────────────────────────────────────────────────────────────
Vocab size:             48,292              60,694
Matched genes:          {len(matched_gene_indices):,}             {census_matched_genes:,}
Gene match rate:        {100*len(matched_gene_indices)/len(gene_names):.1f}%              66.4%
Expression captured:    {pct_matched.mean():.2f}%             {census_expr_matched:.2f}%
Expression lost:        {pct_unmatched.mean():.2f}%              {census_expr_unmatched:.2f}%
─────────────────────────────────────────────────────────────────────
Improvement (Census):   +{census_matched_genes - len(matched_gene_indices)} genes, +{census_expr_matched - pct_matched.mean():.2f}% expression
─────────────────────────────────────────────────────────────────────
""")

# Final recommendation
print("\n" + "=" * 70)
print("RECOMMENDATION")
print("=" * 70)

if census_expr_matched > pct_matched.mean():
    print(f"""
✓ USE CENSUS VOCAB (default_census_vocab.json)
  - Captures {census_expr_matched:.2f}% vs {pct_matched.mean():.2f}% expression
  - {census_matched_genes - len(matched_gene_indices)} more genes matched
  - Better coverage of versioned gene names (AL/AC/AP.N)
""")
else:
    print("→ Both vocabs perform similarly, either is acceptable")

print("=" * 70)

Step 1: Loading Original (default_gene_vocab) mapping
Total genes: 36601
Matched genes (Original): 23536

Step 2: Finding ALL Oli cells
Total Oli cells to analyze: 142,064

Step 3: Analyzing ALL 142,064 Oli cells
  Processing cell 0/142,064 (0.0%)...
  Processing cell 10,000/142,064 (7.0%)...
  Processing cell 20,000/142,064 (14.1%)...
  Processing cell 30,000/142,064 (21.1%)...
  Processing cell 40,000/142,064 (28.2%)...
  Processing cell 50,000/142,064 (35.2%)...
  Processing cell 60,000/142,064 (42.2%)...
  Processing cell 70,000/142,064 (49.3%)...
  Processing cell 80,000/142,064 (56.3%)...
  Processing cell 90,000/142,064 (63.4%)...
  Processing cell 100,000/142,064 (70.4%)...
  Processing cell 110,000/142,064 (77.4%)...
  Processing cell 120,000/142,064 (84.5%)...
  Processing cell 130,000/142,064 (91.5%)...
  Processing cell 140,000/142,064 (98.5%)...
  Processing cell 142,064/142,064 (100%)... Done!

RESULTS: Original Vocab (default_gene_vocab) - ALL Oli Cells

EXPRESSION CONTR

In [6]:
#!/usr/bin/env python3
"""
Compare which SEAAD genes are uniquely matched by Census vs Original vocab
"""

import pandas as pd

# =============================================================================
# PATHS
# =============================================================================
ORIGINAL_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_result.csv"
CENSUS_MAPPING_PATH = "/home/ye/ml-experiments/zhaozhuoresearch/quiz/research-project-gene-ml/gene_mapping_census_vocab.csv"

# =============================================================================
# Load mappings
# =============================================================================
print("=" * 70)
print("Loading mappings")
print("=" * 70)

original_df = pd.read_csv(ORIGINAL_MAPPING_PATH)
census_df = pd.read_csv(CENSUS_MAPPING_PATH)

original_matched = set(original_df[original_df['status'] == 'matched']['seaad_symbol'])
census_matched = set(census_df[census_df['status'] == 'matched']['seaad_symbol'])

print(f"Original vocab matched: {len(original_matched):,}")
print(f"Census vocab matched: {len(census_matched):,}")

# =============================================================================
# Find differences
# =============================================================================
print("\n" + "=" * 70)
print("Comparing matched genes")
print("=" * 70)

# Genes matched by BOTH
both_matched = original_matched & census_matched

# Genes matched ONLY by Census (not Original)
census_only = census_matched - original_matched

# Genes matched ONLY by Original (not Census)
original_only = original_matched - census_matched

print(f"""
VENN DIAGRAM:
─────────────────────────────────────────────────────────────────────
  Matched by BOTH:           {len(both_matched):,}
  Matched ONLY by Census:    {len(census_only):,}
  Matched ONLY by Original:  {len(original_only):,}
─────────────────────────────────────────────────────────────────────
""")

# =============================================================================
# Genes matched ONLY by Census
# =============================================================================
print("\n" + "=" * 70)
print(f"GENES MATCHED ONLY BY CENSUS VOCAB ({len(census_only)} genes)")
print("=" * 70)

census_only_sorted = sorted(census_only)
print(f"\nFirst 50 genes:")
for i, gene in enumerate(census_only_sorted[:50]):
    print(f"  {i+1:3d}. {gene}")

if len(census_only) > 50:
    print(f"\n  ... and {len(census_only) - 50} more")

# Categorize
import re
versioned = [g for g in census_only if re.search(r'\.\d+$', g)]
linc = [g for g in census_only if g.startswith('LINC')]
al_ac_ap = [g for g in census_only if re.match(r'^(AL|AC|AP)\d+', g)]
other = [g for g in census_only if g not in versioned and g not in linc and not re.match(r'^(AL|AC|AP)\d+', g)]

print(f"\nBy category:")
print(f"  Versioned (.N suffix): {len(versioned)}")
print(f"  AL/AC/AP (novel):      {len(al_ac_ap)}")
print(f"  LINC (lncRNA):         {len(linc)}")
print(f"  Other well-annotated:  {len(other)}")

if other:
    print(f"\n  'Other' genes (well-annotated, no version):")
    for gene in sorted(other)[:30]:
        print(f"    - {gene}")

# =============================================================================
# Genes matched ONLY by Original
# =============================================================================
print("\n" + "=" * 70)
print(f"GENES MATCHED ONLY BY ORIGINAL VOCAB ({len(original_only)} genes)")
print("=" * 70)

if len(original_only) == 0:
    print("  None! Census vocab is a superset for SEAAD genes.")
else:
    original_only_sorted = sorted(original_only)
    print(f"\nAll {len(original_only)} genes:")
    for i, gene in enumerate(original_only_sorted):
        print(f"  {i+1:3d}. {gene}")

# =============================================================================
# Summary
# =============================================================================
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"""
─────────────────────────────────────────────────────────────────────
                            Original        Census
─────────────────────────────────────────────────────────────────────
Total matched:              {len(original_matched):,}          {len(census_matched):,}
Unique to this vocab:       {len(original_only):,}              {len(census_only):,}
Shared with other vocab:    {len(both_matched):,}          {len(both_matched):,}
─────────────────────────────────────────────────────────────────────
""")

if len(original_only) == 0:
    print("✓ Census vocab matches ALL genes that Original vocab matches")
    print(f"  PLUS {len(census_only)} additional genes")
    print("\n→ Census vocab is strictly better for SEAAD data")
elif len(census_only) > len(original_only):
    print(f"✓ Census vocab matches {len(census_only) - len(original_only)} more genes net")
    print("\n→ Census vocab is recommended")
else:
    print("→ Original vocab may be preferable")

print("=" * 70)

Loading mappings
Original vocab matched: 23,536
Census vocab matched: 24,285

Comparing matched genes

VENN DIAGRAM:
─────────────────────────────────────────────────────────────────────
  Matched by BOTH:           23,479
  Matched ONLY by Census:    806
  Matched ONLY by Original:  57
─────────────────────────────────────────────────────────────────────


GENES MATCHED ONLY BY CENSUS VOCAB (806 genes)

First 50 genes:
    1. AC000050.1
    2. AC000067.1
    3. AC000124.1
    4. AC002044.1
    5. AC002057.1
    6. AC002057.2
    7. AC002066.1
    8. AC002306.1
    9. AC002386.1
   10. AC002401.1
   11. AC002511.2
   12. AC002551.1
   13. AC003005.2
   14. AC003009.1
   15. AC003092.1
   16. AC003092.2
   17. AC003101.1
   18. AC003958.2
   19. AC003984.1
   20. AC003985.1
   21. AC003988.1
   22. AC004009.1
   23. AC004012.1
   24. AC004047.1
   25. AC004054.1
   26. AC004063.1
   27. AC004069.1
   28. AC004257.1
   29. AC004448.5
   30. AC004490.1
   31. AC004637.1
   32. AC004704.1
